# 오늘도 머신러닝입니다. 예.
- 사실 저는 와인을 좋아하지는 않습니다.
- 근데 술중에서 그나마 마시는게 과일향 술입니다. 외인도 과일주잖아요? 주면 먹긴 먹어요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA # 주성분분석
from sklearn.preprocessing import StandardScaler # 마 서케일러다 안카요
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, confusion_matrix, ConfusionMatrixDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor # VIF
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor # 정말 뜬금없이 XGBoost
import shap

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="viridis", style="darkgrid", font_scale=1)
sns.color_palette("viridis", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Limgul 13'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")

print("Path to dataset files:", path)

In [ ]:
wine = pd.read_csv(f'{path}/winequality-red.csv')

# 정보 확인
## .info()

In [ ]:
wine.info()

- 오. 결측값이 없네.

## .describe()

In [ ]:
wine.describe()

## .isna().sum()

In [ ]:
wine.isna().sum()

## .columns

In [ ]:
wine.columns

## .head()

In [ ]:
wine.head()

# 전처리
## 스케일러어엉
- 이거 PCA 들어갈거라 스케일러좀 쓸게요

In [ ]:
wine_x = wine.copy()
X = wine_x.drop('quality', axis=1)
y = wine['quality']

# 1. 스케일링 (PCA 전 필수!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. PCA 객체 생성 (일단 모든 성분을 다 뽑아봅니다)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

하아니 왜 갑자기 머신러닝 하다 말고 주성분분석이 나옵니까!! 쟤가 칼럼이 12갭니다. 퀄리티가 종속이고 나머지가 독립인데... 그죠. 독립만 11개죠. 그래서 왜 주성분분석을 하느냐... 압축하려고 합니다 압축하려고. 저거 그대로 때려박았다가 다중공선성 터질 수도 있고 일일이 넣었다 뺐다 하는것도 귀찮음.

### Scree plot

In [ ]:
plt.plot(range(1, len(individual_var)+1), individual_var, marker='o')
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")
plt.show()

In [ ]:
plt.plot(range(1, len(cum_var)+1), cum_var, marker='o')
plt.axhline(0.85, linestyle='--')
plt.xlabel("Principal Component")
plt.ylabel("Cumulative Variance")
plt.title("Cumulative Explained Variance")
plt.show()

- 혹시 엘보는 착한 사람만 보이는거냐고요? 저도 안보입니다. 

In [ ]:
# 전체 성분(11개)에 대해 PCA 수행
pca_full = PCA().fit(X_scaled)

# 개별 분산 설명력
individual_var = pca_full.explained_variance_ratio_
# 누적 분산 설명력
cum_var = np.cumsum(individual_var)

# 결과 출력
for i, (ind, cum) in enumerate(zip(individual_var, cum_var)):
    print(f"PC{i+1}: 개별 {ind:.2f} / 누적 {cum:.2f}")
    if cum >= 0.85 and i > 0 and cum_var[i-1] < 0.85:
        print(f"--- 여기까지 딱 끊으면 정보의 {cum*100:.1f}%가 보존됩니다! ---")

- 제 6 주성분까지 채용하면 85% 이상의 설명력을 갖는다는 얘깁니다.

In [ ]:
# 예: 6개가 85% 지점이라면
n_comp = np.argmax(cum_var >= 0.85) + 1
pca_final = PCA(n_components=n_comp)
X_pca_final = pca_final.fit_transform(X_scaled)

print(f"원본 데이터 형태: {X_scaled.shape}")
print(f"PCA 압축 후 데이터 형태: {X_pca_final.shape}")

In [ ]:
for i in range(n_comp):

    loading_scores = pd.Series(
        pca_full.components_[i],
        index=X.columns
    )

    sorted_loadings = loading_scores.abs().sort_values(ascending=False)

    print(f"\nPC{i+1} 핵심 변수 TOP3")
    print(sorted_loadings.head(3))

- 이 에미나이 변수 압축하려고 주성분분석 한댔는데 산점도 줄 때부터 알아봤어야 했는데... (결국 지피티 시킴)
1. 제 1주성분: fixed acidity, citric acid, pH
2. 제 2주성분: total sulfur dioxide, free sulfur dioxide, alcohol
3. 제 3주성분: alcohol, volatile acidity, free sulfur dioxide
4. 제 4주성분: chlorides, sulphates, residual sugar
5. 제 5주성분: residual sugar, alcohol, pH
6. 제 6주성분: pH, volatile acidity, density

In [ ]:
pc_vars = [
    "fixed acidity", "citric acid", "pH",
    "total sulfur dioxide", "free sulfur dioxide", "alcohol",
    "alcohol", "volatile acidity", "free sulfur dioxide",
    "chlorides", "sulphates", "residual sugar",
    "residual sugar", "alcohol", "pH",
    "pH", "volatile acidity", "density"
]

var_set = set(pc_vars)

print(var_set, len(var_set))

- 나 뭐한거임? 다 쳐내려고 했는데 왜 다 안쳐내짐???

### 산점도

In [ ]:
# 2차원으로 축소하여 시각화
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca_2[:, 0], X_pca_2[:, 1], c=y, cmap='plasma', alpha=0.6)
plt.colorbar(scatter, label='Quality')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('PCA 2D Projection of Red Wine Dataset')
plt.show()

# 회기는 역이고 회귀가 분석입니다
- 경희대 파전 맛있는데 많아요 

In [ ]:
X = X_scaled # 저기 주성분 돌리기 전에 거쳤어요 스케일러

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

In [ ]:
print("MAE:", mean_absolute_error(y, pred))
print("MSE:", mean_squared_error(y, pred))
rmse = np.sqrt(mean_squared_error(y, pred))
print("RMSE:", rmse)
print("R²:", r2_score(y, pred))

In [ ]:
plt.scatter(pred, y - pred, alpha=0.5)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

In [ ]:
X = X_pca_final # PCA 돌린거

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

In [ ]:
print("MAE:", mean_absolute_error(y, pred))
print("MSE:", mean_squared_error(y, pred))
rmse = np.sqrt(mean_squared_error(y, pred))
print("RMSE:", rmse)
print("R²:", r2_score(y, pred))

In [ ]:
plt.scatter(pred, y - pred, alpha=0.5)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

## VIF(다중공선성)

In [ ]:
wine_x = wine_x.drop('quality', axis=1)

In [ ]:
# X는 독립변수 데이터프레임 (스케일링 안 해도 되지만 보통 해도 상관없음)
X_vif = pd.DataFrame()
X_vif["variable"] = wine_x.columns
X_vif["VIF"] = [variance_inflation_factor(wine_x.values, i) for i in range(wine_x.shape[1])]

print(X_vif.sort_values(by="VIF", ascending=False))

### 품질과의 상관관계

In [ ]:
wine.corr()['quality'].sort_values(ascending=False)

### 산점도오옹

In [ ]:
plt.figure()

scatter = plt.scatter(
    wine['alcohol'],
    wine['quality'],
    c=wine['quality'],
    cmap='viridis',
    alpha=0.6
)

plt.xlabel('Alcohol')
plt.ylabel('Quality')
plt.title('Alcohol vs Wine Quality')

plt.colorbar(scatter, label='Quality')
plt.show()

### 빡스플롯

In [ ]:
plt.figure()
sns.boxplot(x='quality', y='alcohol', data=wine, hue = 'quality')
plt.title('Alcohol vs Wine Quality')
plt.show()

- 이사람들아 그거 알콜 거 많이 들어가야 소독용 에탄올이여 걍

# 부록: XGBoost가 왜 거기서 나와?
- 회귀하다말고 뭔 분류하는 애가 나오냐고요? 이 데이터셋으로 분류도 되거든요.

In [ ]:
# 일단 쨈
wine_df = wine.copy()
X = wine_df.drop("quality", axis=1)
y = wine_df["quality"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

xgb_model.fit(X_train, y_train)

pred = xgb_model.predict(X_test)

print("R2:", r2_score(y_test, pred))

- 분류가 더 성적이 좋다...ㅋㅋㅋㅋㅋㅋ

In [ ]:
y_pred = xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
residuals = y_test - y_pred

plt.scatter(y_pred, residuals)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.show()

In [ ]:
plt.scatter(y_test, y_pred)
plt.plot([3,8],[3,8])
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

## 피쳐피쳐 베이베베이베

In [ ]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

In [ ]:
importance.plot(kind="bar")
plt.title("Feature Importance")
plt.show()

## SHAP
- 그 이제 XGBoost가 분류를 했잖아요. 그러면 얘가 분류한 기준이 있다 이겁니다. 근데 말을 안해줘요.
- 이친구는 ~~오은영박사님에 빙의해서~~ XGBoost 모델한테 왜 이걸 이렇게 분류한건지 물어봐주는 애입니다. 그 결과가 아래 그래프고요.

In [ ]:
# 1. 모델의 predict 함수를 직접 전달
# 2. masker를 사용하여 데이터의 통계적 분포를 SHAP에게 알려줍니다.
masker = shap.maskers.Independent(data=X_test)
explainer = shap.Explainer(xgb_model.predict, masker)

# 3. SHAP 값 계산 (Permutation 방식은 속도는 좀 걸리지만 매우 정확합니다)
shap_values = explainer(X_test)

# 4. 시각화
shap.summary_plot(shap_values, X_test)